In [1]:
import pandas as pd
df = pd.read_csv(r'C:\Users\anjal\PyTorch\RNN\data\100_Unique_QA_Dataset.csv')

In [2]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
#tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace('?','')
    text = text.replace("'","")

    return text.split()

In [4]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [5]:
#vocab
vocab = {'<UNK>':0}
def build_vocab(row):
    tokenized_ques = tokenize(row['question'])
    tokenized_ans = tokenize(row['answer'])
    print(tokenized_ques,tokenized_ans)

    merged_tokens = tokenized_ques + tokenized_ans

    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

In [6]:
df.apply(build_vocab,axis = 1)

['what', 'is', 'the', 'capital', 'of', 'france'] ['paris']
['what', 'is', 'the', 'capital', 'of', 'germany'] ['berlin']
['who', 'wrote', 'to', 'kill', 'a', 'mockingbird'] ['harper-lee']
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system'] ['jupiter']
['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius'] ['100']
['who', 'painted', 'the', 'mona', 'lisa'] ['leonardo-da-vinci']
['what', 'is', 'the', 'square', 'root', 'of', '64'] ['8']
['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold'] ['au']
['which', 'year', 'did', 'world', 'war', 'ii', 'end'] ['1945']
['what', 'is', 'the', 'longest', 'river', 'in', 'the', 'world'] ['nile']
['what', 'is', 'the', 'capital', 'of', 'japan'] ['tokyo']
['who', 'developed', 'the', 'theory', 'of', 'relativity'] ['albert-einstein']
['what', 'is', 'the', 'freezing', 'point', 'of', 'water', 'in', 'fahrenheit'] ['32']
['which', 'planet', 'is', 'known', 'as', 'the', 'red', 'planet'] ['mars']
['who', 'is', 'the', 'auth

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [7]:
len(vocab)

324

In [8]:
def text_to_indices(text,vocab):

    indexed_text = []
    
    for token in tokenize(text):
    
        if token in vocab:
    
            indexed_text.append(vocab[token])

        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text


In [9]:
text_to_indices('what is campusx',vocab)

[1, 2, 0]

In [10]:
import torch
from torch.utils.data import Dataset,DataLoader

In [11]:
class QADataset(Dataset):

    def __init__(self,df,vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        numerical_ques = text_to_indices(self.df.iloc[index]['question'],self.vocab)
        numerical_ans = text_to_indices(self.df.iloc[index]['answer'],self.vocab)

        return torch.tensor(numerical_ques) , torch.tensor(numerical_ans)
    
    


In [12]:
dataset = QADataset(df,vocab)


In [13]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [43]:
for question,answer in dataloader:
    print(question,answer[0])

tensor([[  1,   2,   3,   4,   5, 279]]) tensor([280])
tensor([[ 42, 137,   2, 138,  39, 139]]) tensor([53])
tensor([[ 78,  79, 195,  81,  19,   3, 196, 197, 198]]) tensor([199])
tensor([[  1,   2,   3, 180, 181, 182, 183]]) tensor([184])
tensor([[  1,   2,   3,   4,   5, 206]]) tensor([207])
tensor([[  1,   2,   3,   4,   5, 286]]) tensor([287])
tensor([[ 42,   2,   3, 210, 137, 168, 211, 169]]) tensor([113])
tensor([[ 1,  2,  3,  4,  5, 53]]) tensor([54])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([179])
tensor([[ 42, 200,   2,  14, 201, 202, 203, 204]]) tensor([205])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([215])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]]) tensor([173])
tensor([[ 1,  2,  3, 24, 25,  5, 26, 19, 27]]) tensor([28])
tensor([[ 10,   2,  62,  63,   3, 283,   5, 284]]) tensor([285])
tensor([[ 42,  18, 118,   3, 186, 187]]) tensor([188])
tensor([[ 78,  79, 288,  81,  19,  14, 289]]) tensor([85])
tensor([[ 78,  79, 150, 

In [15]:
import torch.nn as nn

In [45]:
class SimpleNN(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn = nn.RNN(50,64,batch_first=True)
        self.fc = nn.Linear(64,vocab_size)

    def forward(self,question):
        embedded_ques = self.embedding(question)
        hidden,final = self.rnn(embedded_ques)
        output = self.fc(final.squeeze(0))

        return output

In [46]:
learning_Rate =0.001
epochs = 20

In [47]:
model = SimpleNN(len(vocab))

In [48]:
loss_fun = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_Rate)

In [51]:
for epoch in range(epochs):
    total_loss = 0

    for question,answer in dataloader:
        
        optimizer.zero_grad()

        y_pred = model(question)

        loss = loss_fun(y_pred,answer[0])

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch: {epoch + 1} , Loss: {total_loss:4f}")

Epoch: 1 , Loss: 519.378357
Epoch: 2 , Loss: 448.356711
Epoch: 3 , Loss: 369.238920
Epoch: 4 , Loss: 310.844673
Epoch: 5 , Loss: 261.504781
Epoch: 6 , Loss: 214.335731
Epoch: 7 , Loss: 171.339973
Epoch: 8 , Loss: 133.884193
Epoch: 9 , Loss: 102.393696
Epoch: 10 , Loss: 78.152705
Epoch: 11 , Loss: 60.366341
Epoch: 12 , Loss: 46.993214
Epoch: 13 , Loss: 37.426974
Epoch: 14 , Loss: 30.248306
Epoch: 15 , Loss: 25.123556
Epoch: 16 , Loss: 21.192792
Epoch: 17 , Loss: 17.856444
Epoch: 18 , Loss: 15.587971
Epoch: 19 , Loss: 13.451882
Epoch: 20 , Loss: 11.732000


In [66]:
def predict(model,question,threshold = 0.5):
    #question to num
    numerical_question = text_to_indices(question,vocab)

    ques_tensor = torch.tensor(numerical_question).unsqueeze(0)

    output = model(ques_tensor)

    #convert logits to probability
    probs = torch.nn.functional.softmax(output,dim=1)

    #index of max prob

    value , index = torch.max(probs,dim=1)

    if value < threshold:
        print("i don't know")

    print(list(vocab.keys())[index])

    


In [67]:
predict(model,"what is the capital of france")

paris


In [68]:
list(vocab.keys())[7]

'paris'

for 1st row


In [23]:
dataset[0][0]

tensor([1, 2, 3, 4, 5, 6])

In [ ]:
x = nn.Embedding(324,embedding_dim=50)

In [26]:
a = x(dataset[0][0])

In [27]:
y = nn.RNN(50,64)

In [33]:
#hidden states
y(a)[0]


tensor([[ 4.5531e-01,  2.3268e-01,  9.4089e-02,  5.1741e-01, -5.2658e-01,
         -3.5494e-01, -7.0642e-02, -9.1619e-01, -1.7136e-01,  3.6377e-01,
         -8.7017e-01,  2.8720e-01,  1.0756e-01,  5.9740e-01,  4.8045e-01,
          2.0109e-01,  1.1480e-02, -1.4310e-01, -4.6883e-01, -2.3786e-01,
          2.4867e-02, -1.3166e-01, -6.4494e-01,  2.2238e-01, -8.7849e-01,
         -7.9328e-01, -6.8489e-02,  7.5874e-02,  6.6657e-01, -5.0952e-01,
          2.0360e-01,  8.1687e-01, -6.7013e-01,  7.2703e-01, -8.8133e-01,
          2.3073e-01,  8.5455e-01,  7.4147e-01, -7.4106e-01,  7.1416e-01,
          5.5177e-01, -7.0384e-01,  4.8917e-01,  3.1991e-01,  7.9207e-01,
         -3.6252e-01, -3.9959e-01,  4.8695e-01, -5.7851e-01,  8.7901e-01,
          6.9156e-01,  7.0370e-01, -4.6707e-01, -3.4202e-02,  3.4319e-01,
          3.8911e-02, -2.9950e-02, -6.0773e-01, -5.1056e-01, -1.7616e-01,
          5.6571e-01,  6.7855e-01,  5.6274e-01,  3.3024e-01],
        [-1.8735e-01,  6.2985e-01, -3.6523e-01, -4

In [35]:
#final output
b = y(a)[1]

In [36]:
z = nn.Linear(64,324)

In [ ]:
z(b).shape

torch.Size([1, 324])